In [8]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
os.listdir('/content/drive/MyDrive/colab')

['placement_dataset.csv']

In [6]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/colab/placement_dataset.csv')

In [17]:
from re import M
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

# ---------------------------------------------------------
# 1. Load Dataset
# ---------------------------------------------------------
df = pd.read_csv("placement_dataset.csv")

# Target column
target_col = "CGPA"

# Check if target exists
if target_col not in df.columns:
    raise ValueError(f"Column '{target_col}' not found in dataset.")

    # Drop unnecessary columns if they exist
    drop_cols = ["StudentID", "CGPA_Tier"]
    drop_cols = [col for col in drop_cols if col in df.columns]

    # Select numeric features only
    X = df.drop(columns=drop_cols + [target_col]).select_dtypes(include=np.number)

    # Fill missing values with median
    X = X.fillna(X.median())

    # Target
    y = df[target_col]

    # ---------------------------------------------------------
    # 2. Train-Test Split
    # ---------------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
        )
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Add bias term
    X_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
    X_test = np.c_[np.ones((X_test.shape[0], 1)), X_test]

    # Convert target to numpy array
    y_train = y_train.values.reshape(-1, 1)
    y_test = y_test.values.reshape(-1, 1)
    # ---------------------------------------------------------
    # 4. Gradient Descent Function
    # ---------------------------------------------------------
    def gradient_descent(X, y, learning_rate=0.01, iterations=1000):

        m, n = X.shape
        theta = np.zeros((n, 1))
        losses = []

        for i in range(iterations):
          predictions = X.dot(theta)
          error = predictions - y
          loss = np.sum(error ** 2) / (2 * m)
          losses.append(loss)
          gradient = X.T.dot(error) / m
          theta = theta - learning_rate * gradient
          if (i + 1) % 100 == 0:
            print(f"Iteration {i+1}: Loss = {loss:.4f}")
            return theta, losses
            theta, losses = gradient_descent(X_train, y_train)
            y_pred = X_test.dot(theta)
            mse = mean_squared_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            print("\nGradient Descent Results")
            print("------------------------")
            print("Weights:")
            print(theta.flatten())

            print(f"\nMean Squared Error : {mse:.4f}")
            print(f"R² Score           : {r2:.4f}")

            # ---------------------------------------------------------
            # 6. Compare with Scikit-Learn
            # ---------------------------------------------------------
            lr = LinearRegression()

            lr.fit(X_train[:, 1:], y_train)  # Remove bias column

            sk_pred = lr.predict(X_test[:, 1:])

            print("\nScikit-Learn Results")
            print("------------------------")
            print(f"R² Score : {r2_score(y_test, sk_pred):.4f}")
            print(f"MSE      : {mean_squared_error(y_test, sk_pred):.4f}")

